# Task 3: Broadcast Join Experiment for Query 3

This experiment joins the original Silver taxi-trips and air-quality Delta tables.

## Connect to Spark

In [9]:
import sys

from pyspark.sql import SparkSession

JOB_PATH = '/opt/spark/jobs'
if JOB_PATH not in sys.path:
    sys.path.insert(0, JOB_PATH)

from analytics.benchmarking import measure_registered_query, rows_match
from analytics.runner import execute_registered_query
from analytics.source_tables import register_broadcast_join_sources

spark = (
    SparkSession.builder
    .master('spark://spark-master:7077')
    .appName('Assignment2-BroadcastJoin-Query3')
    .getOrCreate()
)
spark.conf.set('spark.sql.session.timeZone', 'UTC')
spark

## Register the original Delta tables

In [10]:
spark.catalog.clearCache()
register_broadcast_join_sources(spark)
print('Registered Silver taxi trips and air quality views.')

Registered Silver taxi trips and air quality views.


## Isolate the join strategy

- Automatic broadcasting is disabled
- AQE is also disabled so it cannot change the join at [runtime](https://spark.apache.org/docs/4.0.4/sql-performance-tuning.html#adaptive-query-execution).
- The forced query uses a broadcast hint.

In [11]:
JOIN_CONFIGS = {
    'spark.sql.adaptive.enabled': 'false',
    'spark.sql.autoBroadcastJoinThreshold': '-1',
}
for key, value in JOIN_CONFIGS.items():
    spark.conf.set(key, value)

JOIN_CONFIGS

{'spark.sql.adaptive.enabled': 'false',
 'spark.sql.autoBroadcastJoinThreshold': '-1'}

## Measure normal and broadcast joins

In [12]:
NORMAL_QUERY = 'broadcast_join_normal'
BROADCAST_QUERY = 'broadcast_join_forced'
REPETITIONS = 9

normal_results = measure_registered_query(spark, NORMAL_QUERY, REPETITIONS)
broadcast_results = measure_registered_query(spark, BROADCAST_QUERY, REPETITIONS)

assert rows_match(
    normal_results['rows'],
    broadcast_results['rows'],
), 'Normal and broadcast results differ.'

print('Normal and broadcast results match within float tolerance.')

Normal and broadcast results match within float tolerance.


## Normal-join physical plan

This indicates normal join was used: `SortMergeJoin`.

In [13]:
execute_registered_query(spark, NORMAL_QUERY).explain('formatted')

== Physical Plan ==
* HashAggregate (20)
+- Exchange (19)
   +- * HashAggregate (18)
      +- * HashAggregate (17)
         +- * HashAggregate (16)
            +- * Project (15)
               +- * SortMergeJoin Inner (14)
                  :- * Sort (5)
                  :  +- Exchange (4)
                  :     +- * Project (3)
                  :        +- * ColumnarToRow (2)
                  :           +- Scan parquet  (1)
                  +- * Sort (13)
                     +- * HashAggregate (12)
                        +- Exchange (11)
                           +- * HashAggregate (10)
                              +- * Project (9)
                                 +- * Filter (8)
                                    +- * ColumnarToRow (7)
                                       +- Scan parquet  (6)


(1) Scan parquet 
Output [2]: [pickup_datetime#14772, pickup_date#14790]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/silver/silver_taxi_trips]
ReadSchema: str

## Broadcast-join physical plan

- These indicate broadcast join was used: `BroadcastHashJoin` and `BroadcastExchange`.

In [14]:
execute_registered_query(spark, BROADCAST_QUERY).explain('formatted')

== Physical Plan ==
* HashAggregate (19)
+- Exchange (18)
   +- * HashAggregate (17)
      +- * HashAggregate (16)
         +- Exchange (15)
            +- * HashAggregate (14)
               +- * Project (13)
                  +- * BroadcastHashJoin Inner BuildRight (12)
                     :- * Project (3)
                     :  +- * ColumnarToRow (2)
                     :     +- Scan parquet  (1)
                     +- BroadcastExchange (11)
                        +- * HashAggregate (10)
                           +- Exchange (9)
                              +- * HashAggregate (8)
                                 +- * Project (7)
                                    +- * Filter (6)
                                       +- * ColumnarToRow (5)
                                          +- Scan parquet  (4)


(1) Scan parquet 
Output [2]: [pickup_datetime#15226, pickup_date#15244]
Batched: true
Location: PreparedDeltaFileIndex [s3a://delta-data/silver/silver_taxi_trips]
ReadSchema

## Benchmark summary

In [15]:
summary = {
    'normal_samples_seconds': [round(value, 3) for value in normal_results['samples_seconds']],
    'normal_median_seconds': round(normal_results['median_seconds'], 3),
    'broadcast_samples_seconds': [round(value, 3) for value in broadcast_results['samples_seconds']],
    'broadcast_median_seconds': round(broadcast_results['median_seconds'], 3),
    'broadcast_speedup_factor': round(normal_results['median_seconds'] / broadcast_results['median_seconds'],2),
}
summary

{'normal_samples_seconds': [14.12,
  11.257,
  10.489,
  10.324,
  11.173,
  10.266,
  10.452,
  10.095,
  10.229],
 'normal_median_seconds': 10.452,
 'broadcast_samples_seconds': [8.327,
  7.979,
  7.846,
  7.616,
  7.777,
  7.499,
  7.396,
  8.099,
  7.526],
 'broadcast_median_seconds': 7.777,
 'broadcast_speedup_factor': 1.34}

## Cleanup

In [16]:
spark.catalog.clearCache()
spark.stop()